In [1]:
import cv2
import os
import glob
from tqdm import tqdm

input_dir = '/kaggle/input/datasets/jorx04/processed-video-dataset-eng-3'
video_files = glob.glob(os.path.join(input_dir, "*.mp4"))

total_seconds = 0
found_count = 0

print(f"Analyzing {len(video_files)} videos...")

for video_path in tqdm(video_files):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        continue
        
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = cap.get(cv2.CAP_PROP_FRAME_COUNT)
    
    if fps > 0:
        duration = frame_count / fps
        total_seconds += duration
        found_count += 1
    
    cap.release()

hours = int(total_seconds // 3600)
minutes = int((total_seconds % 3600) // 60)
seconds = int(total_seconds % 60)

print("\n" + "="*40)
print(f"RESULTS FOR {found_count} VIDEOS")
print("="*40)
print(f"Total Duration: {hours}h {minutes}m {seconds}s")
print(f"Total Seconds:  {total_seconds:.2f}")
print(f"Average Length: {total_seconds/found_count:.2f} seconds")
print("="*40)

Analyzing 1755 videos...


100%|██████████| 1755/1755 [00:28<00:00, 62.63it/s]


RESULTS FOR 1755 VIDEOS
Total Duration: 2h 27m 46s
Total Seconds:  8866.52
Average Length: 5.05 seconds


In [2]:
from huggingface_hub import hf_hub_download
from kaggle_secrets import UserSecretsClient
import os

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("huggingface")

print("Downloading the SPECTRE environment. This will take a few minutes...")

file_path = hf_hub_download(
    repo_id="Yousef-0541/spectre-env",
    repo_type="dataset",
    filename="spectre_env.tar.gz",
    token=hf_token,
    local_dir="/kaggle/working/"
)

print(f"Download complete! Saved to {file_path}")

spectre_env.tar.gz:   0%|          | 0.00/7.05G [00:00<?, ?B/s]

Download complete! Saved to /kaggle/working/spectre_env.tar.gz


In [3]:
%%bash
cd /kaggle/working/
tar -xzf spectre_env.tar.gz
rm spectre_env.tar.gz
mkdir /kaggle/working/flame_parameters 

In [4]:
%%bash
source /kaggle/working/miniconda/etc/profile.d/conda.sh
conda activate spectre

pip install "numpy==1.23.5" decord

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 112.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 109.3 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 1.24.3
    Uninstalling numpy-1.24.3:
      Successfully uninstalled numpy-1.24.3


In [5]:
%%writefile /kaggle/working/spectre/mass_process.py
import os, sys, glob, argparse, time, collections
import torch
import numpy as np
import cv2
cv2.setNumThreads(0)
from tqdm import tqdm
from datasets.data_utils import landmarks_interpolate
from src.spectre import SPECTRE
from config import cfg as spectre_cfg
from decord import VideoReader, cpu

def extract_frames(vr, face_tracker, stride=1):
    frames_list = vr.get_batch(range(len(vr))).asnumpy() 
    fps = vr.get_avg_fps()
    face_info = collections.defaultdict(list)
    last_faces, last_lmks, last_scores = None, None, None
    for i in range(len(frames_list)):
        if i % stride == 0 or last_faces is None:
            image_bgr = cv2.cvtColor(frames_list[i], cv2.COLOR_RGB2BGR)
            detected_faces = face_tracker.face_detector(image_bgr, rgb=False)
            landmarks, scores = face_tracker.landmark_detector(image_bgr, detected_faces, rgb=False)
            last_faces, last_lmks, last_scores = detected_faces, landmarks, scores
        face_info['bbox'].append(last_faces)
        face_info['landmarks'].append(last_lmks)
        face_info['landmarks_scores'].append(last_scores)
    from external.Visual_Speech_Recognition_for_Multiple_Languages.tracker.utils import get_landmarks
    landmarks = get_landmarks(face_info)
    return frames_list, landmarks, fps

def crop_face_cv2(frame, landmarks, scale=1.0):
    image_size = 224
    left, right = np.min(landmarks[:, 0]), np.max(landmarks[:, 0])
    top, bottom = np.min(landmarks[:, 1]), np.max(landmarks[:, 1])
    old_size = (right - left + bottom - top) / 2
    center = np.array([right - (right - left) / 2.0, bottom - (bottom - top) / 2.0])
    size = int(old_size * scale)
    src_pts = np.array([[center[0] - size / 2, center[1] - size / 2], [center[0] - size / 2, center[1] + size / 2], [center[0] + size / 2, center[1] - size / 2]], dtype=np.float32)
    dst_pts = np.array([[0, 0], [0, image_size - 1], [image_size - 1, 0]], dtype=np.float32)
    tform = cv2.getAffineTransform(src_pts, dst_pts)
    cropped_image = cv2.warpAffine(frame, tform, (image_size, image_size), flags=cv2.INTER_LINEAR)
    return (cropped_image.astype(np.float32) / 255.0).transpose(2, 0, 1)

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('-i', '--input_dir', required=True, type=str)
    parser.add_argument('-o', '--output', required=True, type=str)
    parser.add_argument('--stride', default=1, type=int)
    parser.add_argument('--total_chunks', default=2, type=int)
    parser.add_argument('--chunk_id', default=0, type=int)
    args = parser.parse_args()

    device = 'cuda:0'
    os.makedirs(args.output, exist_ok=True)
    video_files = sorted(glob.glob(os.path.join(args.input_dir, "*.mp4")))
    
    if len(video_files) == 0: return

    chunk_size = int(np.ceil(len(video_files) / float(args.total_chunks)))
    start_idx = args.chunk_id * chunk_size
    end_idx = min(start_idx + chunk_size, len(video_files))
    video_chunk = video_files[start_idx:end_idx]

    print(f"\n[Worker {args.chunk_id}] Assigned {len(video_chunk)} videos (Indices {start_idx} to {end_idx-1})")

    spectre_cfg.pretrained_modelpath = "pretrained/spectre_model.tar"
    spectre_cfg.model.use_tex = False
    spectre = SPECTRE(spectre_cfg, device)
    spectre.eval()

    from external.Visual_Speech_Recognition_for_Multiple_Languages.tracker.face_tracker import FaceTracker
    face_tracker = FaceTracker()
    if hasattr(face_tracker.face_detector, 'device'):
        face_tracker.face_detector.device = torch.device(device)

    for idx, video_path in enumerate(video_chunk):
        video_name = os.path.splitext(os.path.basename(video_path))[0]
        pt_path = os.path.join(args.output, f"{video_name}_flame_params.pt")
        if os.path.exists(pt_path): continue
            
        t0 = time.time()
        try:
            vr = VideoReader(video_path, ctx=cpu(0))
            frames_list, landmarks, fps = extract_frames(vr, face_tracker, stride=args.stride)
            if landmarks is None: continue
                
            landmarks = landmarks_interpolate(landmarks)
            frames_list = list(frames_list)
            frames_list.insert(0, frames_list[0]); frames_list.insert(0, frames_list[0])
            frames_list.append(frames_list[-1]); frames_list.append(frames_list[-1])
            landmarks = list(landmarks)
            landmarks.insert(0, landmarks[0]); landmarks.insert(0, landmarks[0])
            landmarks.append(landmarks[-1]); landmarks.append(landmarks[-1])
            landmarks = np.array(landmarks)
            
            L = 50 
            indices = list(range(len(frames_list)))
            overlapping_indices = [indices[i: i + L] for i in range(0, len(indices), L-4)]
            if len(overlapping_indices[-1]) < 5:
                overlapping_indices[-2] = overlapping_indices[-2] + overlapping_indices[-1]
                overlapping_indices[-2] = np.unique(overlapping_indices[-2]).tolist()
                overlapping_indices = overlapping_indices[:-1]
            overlapping_indices = np.array(overlapping_indices)
            
            all_exp, all_pose, all_shape_params, all_cam = [], [], [], []

            with torch.inference_mode():
                for chunk_id in range(len(overlapping_indices)):
                    frames_chunk = [frames_list[i] for i in overlapping_indices[chunk_id]]
                    landmarks_chunk = landmarks[overlapping_indices[chunk_id]]
                    images_list = [crop_face_cv2(frames_chunk[j], landmarks_chunk[j], scale=1.6) for j in range(len(frames_chunk))]
                    images_array = torch.from_numpy(np.array(images_list)).type(dtype=torch.float32).to(device)

                    codedict, initial_deca_exp, initial_deca_jaw = spectre.encode(images_array)
                    codedict['exp'] = codedict['exp'] + initial_deca_exp
                    codedict['pose'][..., 3:] = codedict['pose'][..., 3:] + initial_deca_jaw

                    for key in codedict.keys():
                        if chunk_id == 0 and chunk_id == len(overlapping_indices) - 1: pass
                        elif chunk_id == 0: codedict[key] = codedict[key][:-2]
                        elif chunk_id == len(overlapping_indices) - 1: codedict[key] = codedict[key][2:]
                        else: codedict[key] = codedict[key][2:-2]

                    all_exp.append(codedict['exp'].detach().cpu())
                    all_pose.append(codedict['pose'].detach().cpu())
                    all_shape_params.append(codedict['shape'].detach().cpu())
                    all_cam.append(codedict['cam'].detach().cpu())

            torch.save({
                'exp': torch.cat(all_exp, dim=0)[2:-2], 
                'pose': torch.cat(all_pose, dim=0)[2:-2],
                'shape': torch.cat(all_shape_params, dim=0)[2:-2], 
                'cam': torch.cat(all_cam, dim=0)[2:-2]
            }, pt_path)
            
            print(f"[Worker {args.chunk_id}] [{idx+1}/{len(video_chunk)}] {video_name} -> {time.time() - t0:.2f}s")
        except Exception as e:
            pass

if __name__ == '__main__':
    main()

Writing /kaggle/working/spectre/mass_process.py


In [6]:
import os
import subprocess
import time

python_path = "/kaggle/working/miniconda/envs/spectre/bin/python"

print("Pre-caching PyTorch Vision Models...")
import torchvision.models as models
model = models.mobilenet_v2(weights='DEFAULT')

output_dir = "/kaggle/working/flame_parameters"
script_path = "/kaggle/working/spectre/mass_process.py"

env0 = os.environ.copy()
env0["CUDA_VISIBLE_DEVICES"] = "0"

env1 = os.environ.copy()
env1["CUDA_VISIBLE_DEVICES"] = "1"

cmd0 = [python_path, "-u", script_path, "--input_dir", input_dir, "--output", output_dir, "--stride", "1", "--total_chunks", "2", "--chunk_id", "0"]
cmd1 = [python_path, "-u", script_path, "--input_dir", input_dir, "--output", output_dir, "--stride", "1", "--total_chunks", "2", "--chunk_id", "1"]

print("\nLaunching Worker 0 on GPU 0...")
print("Launching Worker 1 on GPU 1...\n")

p0 = subprocess.Popen(cmd0, env=env0, cwd="/kaggle/working/spectre")
time.sleep(20)
p1 = subprocess.Popen(cmd1, env=env1, cwd="/kaggle/working/spectre")

try:
    p0.wait()
    p1.wait()
except KeyboardInterrupt:
    p0.terminate()
    p1.terminate()

print("\nBoth GPUs have successfully finished processing!")

Pre-caching PyTorch Vision Models...
Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-7ebf99e0.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 118MB/s] 



Launching Worker 0 on GPU 0...
Launching Worker 1 on GPU 1...


[Worker 0] Assigned 878 videos (Indices 0 to 877)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 192MB/s]
Downloading: "https://github.com/pytorch/vision/archive/v0.8.1.zip" to /root/.cache/torch/hub/v0.8.1.zip
Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth
100%|██████████| 13.6M/13.6M [00:00<00:00, 114MB/s] 
/kaggle/working/miniconda/envs/spectre/lib/python3.8/site-packages/pytorch3d/io/obj_io.py:548: UserWarning: Mtl file does not exist: /kaggle/working/spectre/data/template.mtl
  warnings.warn(f"Mtl file does not exist: {f}")



[Worker 1] Assigned 877 videos (Indices 878 to 1754)


Using cache found in /root/.cache/torch/hub/pytorch_vision_v0.8.1
/kaggle/working/miniconda/envs/spectre/lib/python3.8/site-packages/pytorch3d/io/obj_io.py:548: UserWarning: Mtl file does not exist: /kaggle/working/spectre/data/template.mtl
  warnings.warn(f"Mtl file does not exist: {f}")
/kaggle/working/spectre/mass_process.py:102: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  overlapping_indices = np.array(overlapping_indices)


[Worker 0] [1/878] -OknSRRyFJE_0 -> 25.28s
[Worker 0] [2/878] -OknSRRyFJE_1 -> 2.53s


/kaggle/working/spectre/mass_process.py:102: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  overlapping_indices = np.array(overlapping_indices)


[Worker 1] [1/877] VcY28eKMM3I_7 -> 17.57s
[Worker 0] [3/878] -OknSRRyFJE_2 -> 9.95s
[Worker 0] [4/878] -OknSRRyFJE_3 -> 5.29s
[Worker 1] [2/877] VcY28eKMM3I_8 -> 9.54s
[Worker 1] [3/877] VcY28eKMM3I_9 -> 6.54s
[Worker 0] [5/878] -OknSRRyFJE_4 -> 12.74s
[Worker 1] [4/877] W8cBDZD7Sx0_0 -> 4.73s
[Worker 1] [5/877] W8cBDZD7Sx0_1 -> 4.12s
[Worker 1] [6/877] YM33XfEZrwc_0 -> 2.57s
[Worker 1] [7/877] YM33XfEZrwc_1 -> 2.90s
[Worker 0] [6/878] -OknSRRyFJE_5 -> 11.12s
[Worker 0] [7/878] -m8tHXm8D8w_0 -> 2.88s
[Worker 1] [8/877] Yqsxc4LUc0k_0 -> 5.50s
[Worker 0] [8/878] -m8tHXm8D8w_1 -> 2.89s
[Worker 1] [9/877] Yqsxc4LUc0k_1 -> 6.03s
[Worker 0] [9/878] -m8tHXm8D8w_10 -> 6.72s
[Worker 1] [10/877] Yqsxc4LUc0k_2 -> 4.40s
[Worker 0] [10/878] -m8tHXm8D8w_11 -> 6.12s
[Worker 1] [11/877] Yqsxc4LUc0k_3 -> 5.90s
[Worker 1] [12/877] Yqsxc4LUc0k_4 -> 2.78s
[Worker 0] [11/878] -m8tHXm8D8w_12 -> 5.12s
[Worker 0] [12/878] -m8tHXm8D8w_13 -> 8.71s
[Worker 1] [13/877] _9zEsVkLC30_0 -> 9.54s
[Worker 1] [14/877] 

In [7]:
%%bash
rm -rf /kaggle/working/.cache
rm -rf /kaggle/working/.virtual_documents
rm -rf /kaggle/working/miniconda
rm -rf /kaggle/working/spectre